In [0]:
spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")


Out[2]: '12.2.x-cpu-ml-scala2.12'

In [0]:
%pip install azure-storage-blob==12.19.1
%pip install lxml_html_clean==0.1.1
%pip install newsapi-python==0.2.7
%pip install newspaper3k==0.2.8
%pip install nltk==3.8.1
%pip install pandas==2.2.1
%pip install psycopg2-binary==2.9.9
%pip install pyarrow==15.0.2
%pip install SQLAlchemy==2.0.20

Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.


In [0]:


# Import necessary libraries
from newsapi.newsapi_client import NewsApiClient
import pandas as pd
from newspaper import Article, Config
from nltk.corpus import stopwords
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datetime import date, timedelta

def extract_transform_function():
    today = date.today()
    # Get today's date
    yesterday = today - timedelta(days = 1)
    # Get yesterday's date
    day_before_yesterday = today - timedelta(days = 2)
    # Get the day before yesterday's date

    # Initialize the News API client with an API key
    newsapi = NewsApiClient(api_key='')

    # Get top headlines for the entertainment category in English, with a page size of 90
    top_headlines = newsapi.get_top_headlines(   
                                            category='entertainment',
                                            language='en',
                                            page_size = 90,
                                            page= 1)

    # Extract articles from the API response
    articles = top_headlines.get('articles',[])

    # Create a DataFrame from the articles, selecting specific columns
    init_df = pd.DataFrame(articles, columns = ['source','title','publishedAt','author','url'])

    # Extract the 'name' field from the 'source' dictionary in each row
    init_df['source'] = init_df['source'].apply(lambda x: x['name'] if pd.notna(x) and 'name' in x else None)

    # Convert 'publishedAt' to datetime format
    init_df['publishedAt'] = pd.to_datetime(init_df['publishedAt'])

    # Filter the DataFrame for articles published on the day before yesterday or yesterday
    filtered_df = init_df[(init_df['publishedAt'].dt.date == day_before_yesterday) | (init_df['publishedAt'].dt.date == yesterday)]
    # Rename the 'publishedAt' column to 'date_posted'
    filtered_df.rename(columns={'publishedAt': 'date_posted'}, inplace=True)

    # Make a copy of the filtered DataFrame
    df = filtered_df.copy()

    # Function to retrieve the full content of an article given its URL
    def full_content(url):
        # Set up the user agent for the browser configuration
        user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36'
        config = Config()
        config.browser_user_agent = user_agent
        page = Article(url, config = config)

        try:
            # Download and parse the article
            page.download()
            page.parse()
            return page.text
        except Exception as e:
            print(f"Error retrieving content from {url}: {e}")
            return 'couldnt retrieve'

    # Apply the full_content function to each URL in the DataFrame
    df['content'] = df['url'].apply(full_content)
    # Replace newlines in the 'content' column with spaces
    df['content'] = df['content'].str.replace('\n', ' ')
    # Filter out rows where the content could not be retrieved
    df = df[df['content'] != 'couldnt retrieve']

    # Download the NLTK stopwords dataset and other required datasets
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('wordnet')

    # Function to count words in a text excluding stopwords
    def count_words_without_stopwords(text):
        if isinstance(text, (str, bytes)):
            words = nltk.word_tokenize(str(text))
            stop_words = set(stopwords.words('english'))
            filtered_words = [word for word in words if word.lower() not in stop_words]
            return len(filtered_words)
        else:
            return 0

    # Apply the word count function to the 'content' column
    df['word_count'] = df['content'].apply(count_words_without_stopwords)

    # Download the VADER sentiment analysis lexicon
    nltk.download('vader_lexicon')

    # Initialize the SentimentIntensityAnalyzer
    sid = SentimentIntensityAnalyzer()

    # Function to get sentiment and compound score for a given text
    def get_sentiment(row):
        sentiment_scores = sid.polarity_scores(row)
        compound_score = sentiment_scores['compound']

        if compound_score >= 0.05:
            sentiment = 'Positive'
        elif compound_score <= -0.05:
            sentiment = 'Negative'
        else:
            sentiment = 'Neutral'

        return sentiment, compound_score

    # Apply the sentiment analysis function to the 'content' column
    df[['sentiment', 'compound_score']] = df['content'].astype(str).apply(lambda x: pd.Series(get_sentiment(x)))

    return df

# Call the extract_transform_function and store the result in a DataFrame
dataframe = extract_transform_function()
print(dataframe)

<command-3856577217755883>:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.rename(columns={'publishedAt': 'date_posted'}, inplace=True)
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


                  source  ... compound_score
0                allkpop  ...         0.6928
1           HotNewHipHop  ...        -0.1406
2          Dark Horizons  ...         0.9136
3             The Ringer  ...        -0.9439
4     Hollywood Reporter  ...        -0.9668
5       Associated Press  ...        -0.3818
6               Page Six  ...         0.9931
7   Entertainment Weekly  ...        -0.9893
8   Entertainment Weekly  ...        -0.9821
9                Variety  ...         0.9921
10          HotNewHipHop  ...         0.9332
11    Hollywood Reporter  ...         0.9996
12             TVInsider  ...         0.8356
13              Page Six  ...         0.9950
14             IndieWire  ...         0.8907
15          Israel Hayom  ...         0.9709
16          Collider.com  ...        -0.9712
17  Entertainment Weekly  ...        -0.9880
18    Hollywood Reporter  ...         0.9981
19           Tom's Guide  ...         0.9891
20          Cartoon Brew  ...         0.8647

[21 rows 

In [0]:
 %sql
CREATE DATABASE IF NOT EXISTS the_news;
CREATE TABLE IF NOT EXISTS the_news.news_table (
source STRING,
title STRING,
date_posted DATE,
author STRING,
url STRING,
content STRING,
word_count INT,
sentiment STRING,
compound_score DOUBLE
)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, DoubleType

# Initialize Spark session
spark = SparkSession.builder.appName("CreateTableExample").getOrCreate()

# Define the schema explicitly if necessary
schema = StructType([
    StructField("source", StringType(), True),
    StructField("title", StringType(), True),
    StructField("date_posted", DateType(), True), 
    StructField("author", StringType(), True),
    StructField("url", StringType(), True),
    StructField("content", StringType(), True),
    StructField("word_count", IntegerType(), True),
    StructField("sentiment", StringType(), True),
    StructField("compound_score", DoubleType(), True)
])

spark_df = spark.createDataFrame(dataframe, schema=schema)
spark_df.write.mode('overwrite').saveAsTable('the_news.news_table')

In [0]:
%sql

SELECT * FROM the_news.news_table;

source title date_posted author url content word_count sentiment compound_score allkpop Additional photos revealed just before Kim Soo Hyun’s press conference - allkpop 2025-03-31 Alec06 https://www.allkpop.com/article/2025/03/additional-photos-revealed-just-before-kim-soo-hyuns-press-conference Actor Kim Soo Hyun's upcoming press conference to address personal life rumors related to the late actress Kim Sae Ron has led YouTube channel 'Garoseo Research Institute' (Gaseyeon) to release additional personal photos of Kim Soo Hyun. On March 31, Gaseyeon posted a photo of Kim Soo Hyun with short hair, along with a caption that read: “Pedo ‘Bul Geung’, Soo Hyun. By the way, what does ‘Bul Geung’ mean? You probably remember what scene this ‘video’ was from, right?” In the photo, Kim Soo Hyun is seen eating while sporting short hair. Gaseyeon continued: “At that time, 11th-grade Kim Sae Ron made chicken stew at 11:20 PM. And at 1 AM that night, the song she sang for him was ‘The One and Only You.’ You sang pretty well, didn’t you? It was the theme song of 'Moon Embracing the Sun', right?” They added, “We will no longer tolerate the harassment of Kim Sae Ron’s family. You’ve already hurt her enough. You’re certainly going to hell... but before you go to a more terrifying hell... make a sincere public apology now.” The channel then issued a warning: “Depending on what you do tomorrow... we will respond accordingly.” Kim Soo Hyun has recently been at the center of controversy over allegations that he had a romantic relationship with Kim Sae Ron when she was a minor. He is scheduled to hold an urgent press conference later today at a location in Seoul to personally address the allegations. According to Kim Sae Ron’s family, based on text messages and handwritten letters, they believe that Kim Sae Ron, then 15, was in a relationship with the then 27-year-old Kim Soo Hyun starting in 2015. However, Kim Soo Hyun’s side claims the relationship began in 2019, after Kim Sae Ron had reached legal adulthood. Amid the dispute, previously unreleased KakaoTalk messages from 2016 and 2018 have surfaced, containing affectionate exchanges such as “I’ll hug you and fall asleep” and “When can I sleep in your arms?” The controversy has further intensified following a statement from the brother of the late singer and actress Sulli. He alleged that Sulli was misled by Kim Soo Hyun and pressured into filming a nude scene in the film 'Real' during her lifetime. 299 Positive 0.6928 HotNewHipHop Kanye West Gives An Explicit Response To Fans Saying He Offend Jay-Z - HotNewHipHop 2025-03-31 Bryson "Boom" Paul https://www.hotnewhiphop.com/898439-kanye-west-explicit-response-jay-z-hip-hop-news It appears that friends have turned into enemies between Kanye West and Jay-Z. In March, Ye shared a series of tweets that included sly remarks about many of his previous collaborators. One of the tweets that went viral was a claim that Jay-Z and Beyonce's children had disabilities. In a new interview with DJ Akademiks, West would addresses the social media backlash he received behind the tweet and berated Jigga somemore. "Everything is like, 'But you offended Jay-Z," Ye said to AK. "F*ck him, you know what I'm sayin." After that, Ye would discuss Jay-Z still making money off of the mogul's catalog. He continued: "Put it like this, let's take it to money, how much money you think Jay-Z make off my catalog versus what I make off it? Next subject." Ye would move on to slamming John Legend and Pusha T. West claims to have changed John Legends life. About Legend, West said, "Look at John Legend Ol Sussy-Ass. I ain't never did nothing to him. Changed his f*ckin life. I changed generations of his life." John Legend would speak on Ye's previous comments in the latest episode of Drink Champs. The singer cleared up the alleged offer of a record deal by Damon Dash and his reveal he has no ties with Kanye West anymore. Kanye West & Jay-Z Jay-Z and Beyonce have reportedly sought out leg